# Neuronal Tuning Summary

This notebook runs the cross-session / cross-condition unit-triage **aggregator** over the pre-computed `triage_stats` block stored in every per-cluster `*_tuning_curves_data.pkl`.

It collapses same-day duplicate units across sessions and conditions into a single pickle keyed by `(animal_id, YYYYMMDD, imec, cluster_id)`, preserving per-session evidence so consistency of tuning can be inspected across repeats. Inputs are session-list `.txt` files (one path per line) and the authoritative `unit_catalog.csv` for anatomy.

The triage stats themselves are produced by `generate-rm` — this notebook is a pure pkl-to-pickle pass and never touches spike or USV data. Thresholds default to `_parameter_settings/analyses_settings.json` (`detect_interesting_tuning_neurons` block) and can be overridden in the configuration cell below.


In [ ]:
### Imports

from usv_playpen.analyses.unit_triage_aggregator import aggregate_units_across_conditions


## Configure thresholds

Thresholds default to the values in `_parameter_settings/analyses_settings.json` (`detect_interesting_tuning_neurons` block); override here if you want to sweep different values across the same set of pkls without re-computing tuning.

In [ ]:
THRESHOLDS = {
    "z_threshold":               3.0,
    "min_consecutive_bins":        3,
    "vmi_alpha":                0.01,
    "vmi_min_bouts":              10,
    "spatial_info_bps_threshold": 0.5,
}


## Cross-session / cross-condition aggregator

Collapses same-day duplicate units across the sessions in `CONDITION_TO_SESSION_LIST` into a single pickle keyed by `(animal_id, YYYYMMDD, imec, cluster_id)`. Each unit carries its identity, the catalog `anatomy_region`, and a `conditions` block — one entry per condition listing, per modality, `n_significant`, `n_tested`, `consistency`, an aggregate scalar, and per-session evidence rows.

Each value of `CONDITION_TO_SESSION_LIST` is a path to a `.txt` file with one session root per line (the `ephys_courtship_*_sessions_list.txt` lists under `/mnt/falkner/Bartul/modeling/input_files/`). Sessions missing a `tuning_curves` directory are recorded under `sessions_skipped` and not counted. Anatomy is read from the authoritative `unit_catalog.csv`; orphan pkls (no catalog row) raise.

The output is written to `<out_dir>/unit_triage_<YYYYMMDD>_<HHMMSS>.pkl` and is the input to all downstream cross-session plotting.

In [ ]:
CONDITION_TO_SESSION_LIST = {
    "intact_female": "/mnt/falkner/Bartul/modeling/input_files/ephys_courtship_intact_partners_sessions_list.txt",
    "mute_female":   "/mnt/falkner/Bartul/modeling/input_files/ephys_courtship_mute_female_sessions_list.txt",
}

CATALOG_PATH = "/mnt/falkner/Bartul/EPHYS/unit_catalog.csv"
AGGREGATOR_OUT_DIR = "/mnt/falkner/Bartul/neuronal_tuning"
DATA_ROOT = "/mnt/falkner/Bartul/Data"


In [ ]:
aggregator_pkl_path = aggregate_units_across_conditions(
    condition_to_session_list=CONDITION_TO_SESSION_LIST,
    catalog_path=CATALOG_PATH,
    out_dir=AGGREGATOR_OUT_DIR,
    data_root=DATA_ROOT,
    z_threshold=THRESHOLDS["z_threshold"],
    min_consecutive_bins=THRESHOLDS["min_consecutive_bins"],
    vmi_alpha=THRESHOLDS["vmi_alpha"],
    vmi_min_bouts=THRESHOLDS["vmi_min_bouts"],
    spatial_info_bps_threshold=THRESHOLDS["spatial_info_bps_threshold"],
    message_output=print,
)

print(f"\nAggregator wrote: {aggregator_pkl_path}")
